# 02 - Preprocessing y Feature Engineering

Este notebook implementa la etapa de transformaciones del dataset de victimas de siniestros viales en CABA. El objetivo es construir una version limpia y enriquecida, reproducible de punta a punta, sin modificar los datos crudos.

## Criterio de arquitectura de datos

Se separa `data/raw/` de `data/processed/` para preservar la fuente original como evidencia inmutable del dato recibido. Esta practica permite auditar decisiones, repetir el pipeline desde cero y comparar resultados si cambian las reglas de limpieza. En esta etapa solo se lee desde `data/raw/` y todo resultado derivado se guarda en `data/processed/`.

In [1]:
from pathlib import Path
import logging
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import cargar_dataset

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
LOG_DIR = PROJECT_ROOT / "logs"
OUTPUT_FILE = PROCESSED_DATA_DIR / "siniestros_limpio_enriquecido.csv"
LOG_FILE = LOG_DIR / "pipeline.log"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger("preprocessing_siniestros")
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(LOG_FILE, mode="w", encoding="utf-8-sig")
file_handler.setFormatter(
    logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s")
)
logger.addHandler(file_handler)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
logger.addHandler(stream_handler)

logger.info("Inicio del preprocessing")

INFO | Inicio del preprocessing


## Carga del dataset fuente

La carga se realiza desde el Excel original disponible en `data/raw/`, preferentemente `siniestros_viales_victimas.xlsx` si existe y, en su defecto, `siniestros.xlsx`. El dataset se copia inmediatamente (`df = df_raw.copy()`) para evitar cualquier modificacion accidental sobre la referencia cargada.

In [2]:
raw_candidates = [
    RAW_DATA_DIR / "siniestros_viales_victimas.xlsx",
    RAW_DATA_DIR / "siniestros.xlsx",
]
raw_file = next((path for path in raw_candidates if path.exists()), None)

if raw_file is None:
    available_files = sorted(
        path for path in RAW_DATA_DIR.iterdir()
        if path.suffix.lower() in {".xlsx", ".xls", ".csv"}
    )
    if not available_files:
        raise FileNotFoundError("No se encontro un archivo Excel o CSV en data/raw/.")
    raw_file = available_files[0]

try:
    logger.info("Carga de datos desde %s", raw_file)
    df_raw = cargar_dataset(raw_file)
    df = df_raw.copy()
    logger.info("Cantidad inicial: %s filas x %s columnas", df.shape[0], df.shape[1])
except Exception:
    logger.exception("Error durante la carga de datos")
    raise

print(f"Archivo fuente: {raw_file.resolve()}")
print(f"Shape inicial: {df.shape}")
df.head()

INFO | Carga de datos desde C:\Users\Germán\Desktop\TP AVANZADA\tp-final-siniestros-viales\data\raw\siniestros.xlsx


Archivo Excel detectado: siniestros.xlsx
Hojas disponibles: ['VICTIMAS', 'DICCIONARIO_VICTIMAS']
Hoja usada: VICTIMAS


INFO | Cantidad inicial: 62076 filas x 9 columnas


Dataset cargado: 62076 filas x 9 columnas
Archivo fuente: C:\Users\Germán\Desktop\TP AVANZADA\tp-final-siniestros-viales\data\raw\siniestros.xlsx
Shape inicial: (62076, 9)


,id_siniestro,fecha_siniestro,anio_siniestro,modo_desplazamiento_victima,sexo_victima,edad_victima,GRAVEdad_victima,rol_victima,fecha_fallecimiento_victima
0,LC-2019-0000647,2019-01-01,2019,MOTO,M,54,GRAVE,SD,NaN
1,LC-2019-0000600,2019-01-01,2019,SD,F,1,LEVE,SD,NaN
2,LC-2019-0000136,2019-01-01,2019,SD,F,21,LEVE,SD,NaN
3,LC-2019-0000082,2019-01-01,2019,SD,F,32,LEVE,SD,NaN
4,LC-2019-0000194,2019-01-01,2019,SD,F,33,LEVE,SD,NaN


## Limpieza y normalizacion

`SD` y valores equivalentes se interpretan como datos faltantes porque en el diccionario del dataset representan ausencia de informacion registrada, no una categoria sustantiva del fenomeno vial. Mantenerlos como categoria podria sesgar conteos y modelos al hacer que la falta de dato parezca una propiedad real de la victima o del siniestro.

Tambien se convierten tipos relevantes, se normalizan categorias textuales con espacios recortados y mayusculas, se revisan duplicados y se eliminan columnas no utilizadas para modelado. `fecha_fallecimiento_victima` se elimina porque se conoce despues del evento y puede inducir data leakage al anticipar informacion directamente asociada al desenlace mortal.

In [3]:
try:
    logger.info("Inicio de limpieza y normalizacion")

    original_columns = df.columns.tolist()
    if "GRAVEdad_victima" in df.columns and "gravedad_victima" not in df.columns:
        df = df.rename(columns={"GRAVEdad_victima": "gravedad_victima"})
        logger.info("Columna GRAVEdad_victima renombrada a gravedad_victima")

    missing_equivalents = {
        "", "SD", "S/D", "SIN DATO", "SIN DATOS", "NO DATA", "N/D", "ND", "NA", "NAN", "NONE", "NULL"
    }

    text_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()
    for column in text_columns:
        normalized = df[column].astype("string").str.strip()
        normalized_upper = normalized.str.upper()
        df[column] = normalized_upper.mask(normalized_upper.isin(missing_equivalents), pd.NA)

    if "edad_victima" in df.columns:
        df["edad_victima"] = pd.to_numeric(df["edad_victima"], errors="coerce")

    if "fecha_siniestro" in df.columns:
        df["fecha_siniestro"] = pd.to_datetime(df["fecha_siniestro"], errors="coerce")

    duplicated_rows = int(df.duplicated().sum())
    logger.info("Filas duplicadas detectadas: %s", duplicated_rows)

    columns_to_drop = [
        column for column in ["id_siniestro", "fecha_fallecimiento_victima"]
        if column in df.columns
    ]
    df = df.drop(columns=columns_to_drop)
    logger.info("Columnas eliminadas para modelado: %s", columns_to_drop)
    logger.info("Transformaciones aplicadas: SD a NaN, edad numerica, fecha datetime, texto strip/upper, revision de duplicados")

except Exception:
    logger.exception("Error durante la limpieza y normalizacion")
    raise

print(f"Columnas originales: {original_columns}")
print(f"Duplicados detectados: {duplicated_rows}")
print(f"Columnas eliminadas: {columns_to_drop}")
print(f"Shape tras limpieza: {df.shape}")
df.head()

INFO | Inicio de limpieza y normalizacion


INFO | Columna GRAVEdad_victima renombrada a gravedad_victima


INFO | Filas duplicadas detectadas: 1166


INFO | Columnas eliminadas para modelado: ['id_siniestro', 'fecha_fallecimiento_victima']


INFO | Transformaciones aplicadas: SD a NaN, edad numerica, fecha datetime, texto strip/upper, revision de duplicados


Columnas originales: ['id_siniestro', 'fecha_siniestro', 'anio_siniestro', 'modo_desplazamiento_victima', 'sexo_victima', 'edad_victima', 'GRAVEdad_victima', 'rol_victima', 'fecha_fallecimiento_victima']
Duplicados detectados: 1166
Columnas eliminadas: ['id_siniestro', 'fecha_fallecimiento_victima']
Shape tras limpieza: (62076, 7)


,fecha_siniestro,anio_siniestro,modo_desplazamiento_victima,sexo_victima,edad_victima,gravedad_victima,rol_victima
0,2019-01-01,2019,MOTO,M,54,GRAVE,<NA>
1,2019-01-01,2019,<NA>,F,1,LEVE,<NA>
2,2019-01-01,2019,<NA>,F,21,LEVE,<NA>
3,2019-01-01,2019,<NA>,F,32,LEVE,<NA>
4,2019-01-01,2019,<NA>,F,33,LEVE,<NA>


## Variables derivadas

Las variables derivadas resumen informacion de alto valor analitico para etapas posteriores. Los grupos etarios reducen ruido y facilitan comparaciones; las variables temporales capturan estacionalidad y patrones semanales; la vulnerabilidad del usuario incorpora conocimiento de dominio sobre exposicion y proteccion relativa en la via publica.

`es_mortal` puede funcionar como target predictivo porque representa un desenlace binario asociado a la severidad del siniestro. Para un segundo enfoque, `es_grave_o_mortal` permite modelar eventos de alta severidad agrupando casos graves y mortales frente a lesiones leves.

In [4]:
def asignar_grupo_edad(edad):
    if pd.isna(edad):
        return "sin_dato"
    if edad < 18:
        return "menor_18"
    if edad <= 30:
        return "18_30"
    if edad <= 45:
        return "31_45"
    if edad <= 60:
        return "46_60"
    if edad <= 75:
        return "61_75"
    return "76_mas"


def asignar_vulnerabilidad(modo):
    alta = {"PEATON", "MOTO", "BICICLETA", "MONOPATIN"}
    media = {"AUTO", "TRANSPORTE PUBLICO", "TAXI", "UTILITARIO", "CAMION"}

    if pd.isna(modo):
        return "DESCONOCIDA"
    if modo in alta:
        return "ALTA"
    if modo in media:
        return "MEDIA"
    return "DESCONOCIDA"

try:
    logger.info("Inicio de feature engineering")

    df["edad_grupo"] = df["edad_victima"].apply(asignar_grupo_edad)

    if "gravedad_victima" not in df.columns:
        raise KeyError("No se encontro la columna gravedad_victima para crear targets.")

    df["es_mortal"] = np.where(df["gravedad_victima"].eq("MORTAL"), 1, 0)
    df["es_grave_o_mortal"] = np.where(df["gravedad_victima"].isin(["GRAVE", "MORTAL"]), 1, 0)

    if "modo_desplazamiento_victima" in df.columns:
        df["vulnerabilidad_usuario"] = df["modo_desplazamiento_victima"].apply(asignar_vulnerabilidad)
    else:
        df["vulnerabilidad_usuario"] = "DESCONOCIDA"

    if "fecha_siniestro" in df.columns:
        df["mes_siniestro"] = df["fecha_siniestro"].dt.month.astype("Int64")
        df["dia_semana_siniestro"] = df["fecha_siniestro"].dt.dayofweek.astype("Int64")
        df["trimestre_siniestro"] = df["fecha_siniestro"].dt.quarter.astype("Int64")
    else:
        df["mes_siniestro"] = pd.Series(pd.NA, index=df.index, dtype="Int64")
        df["dia_semana_siniestro"] = pd.Series(pd.NA, index=df.index, dtype="Int64")
        df["trimestre_siniestro"] = pd.Series(pd.NA, index=df.index, dtype="Int64")

    logger.info("Variables creadas: edad_grupo, es_mortal, es_grave_o_mortal, vulnerabilidad_usuario, mes/dia/trimestre")

except Exception:
    logger.exception("Error durante el feature engineering")
    raise

print(f"Shape tras feature engineering: {df.shape}")
df.head()

INFO | Inicio de feature engineering


INFO | Variables creadas: edad_grupo, es_mortal, es_grave_o_mortal, vulnerabilidad_usuario, mes/dia/trimestre


Shape tras feature engineering: (62076, 14)


,fecha_siniestro,anio_siniestro,modo_desplazamiento_victima,sexo_victima,edad_victima,gravedad_victima,rol_victima,edad_grupo,es_mortal,es_grave_o_mortal,vulnerabilidad_usuario,mes_siniestro,dia_semana_siniestro,trimestre_siniestro
0,2019-01-01,2019,MOTO,M,54,GRAVE,<NA>,46_60,0,1,ALTA,1,1,1
1,2019-01-01,2019,<NA>,F,1,LEVE,<NA>,menor_18,0,0,DESCONOCIDA,1,1,1
2,2019-01-01,2019,<NA>,F,21,LEVE,<NA>,18_30,0,0,DESCONOCIDA,1,1,1
3,2019-01-01,2019,<NA>,F,32,LEVE,<NA>,31_45,0,0,DESCONOCIDA,1,1,1
4,2019-01-01,2019,<NA>,F,33,LEVE,<NA>,31_45,0,0,DESCONOCIDA,1,1,1


## Validacion del resultado

Antes de persistir el archivo final se revisan dimensiones, tipos, nulos y distribuciones de las variables nuevas. Estas salidas dejan evidencia ejecutable de que el dataset enriquecido conserva consistencia estructural y de que los targets quedaron definidos.

In [5]:
new_columns = [
    "edad_grupo",
    "es_mortal",
    "es_grave_o_mortal",
    "vulnerabilidad_usuario",
    "mes_siniestro",
    "dia_semana_siniestro",
    "trimestre_siniestro",
]

print("Shape final:", df.shape)
print("\nTipos finales:")
display(df.dtypes.rename("dtype").to_frame())

print("\nNulos finales:")
display(df.isna().sum().rename("nulos").to_frame())

for column in new_columns:
    print(f"\nValue counts - {column}:")
    display(df[column].value_counts(dropna=False).rename("cantidad").to_frame())

print("\nDistribucion target es_mortal:")
display(df["es_mortal"].value_counts(normalize=False).rename("cantidad").to_frame())
display(df["es_mortal"].value_counts(normalize=True).rename("proporcion").to_frame())

print("\nDistribucion target es_grave_o_mortal:")
display(df["es_grave_o_mortal"].value_counts(normalize=False).rename("cantidad").to_frame())
display(df["es_grave_o_mortal"].value_counts(normalize=True).rename("proporcion").to_frame())

Shape final: (62076, 14)

Tipos finales:


,dtype
fecha_siniestro,datetime64[ns]
anio_siniestro,int64
modo_desplazamiento_victima,string[python]
sexo_victima,string[python]
edad_victima,Int64
gravedad_victima,string[python]
rol_victima,string[python]
edad_grupo,object
es_mortal,int64
es_grave_o_mortal,int64



Nulos finales:


,nulos
fecha_siniestro,0
anio_siniestro,0
modo_desplazamiento_victima,21874
sexo_victima,11054
edad_victima,16756
gravedad_victima,0
rol_victima,50203
edad_grupo,0
es_mortal,0
es_grave_o_mortal,0



Value counts - edad_grupo:


...

,cantidad
es_mortal,
0,61466
1,610


,proporcion
es_mortal,
0,0.990173
1,0.009827



Distribucion target es_grave_o_mortal:


,cantidad
es_grave_o_mortal,
0,58995
1,3081


,proporcion
es_grave_o_mortal,
0,0.950367
1,0.049633


## Guardado del dataset procesado

El archivo final se guarda como `data/processed/siniestros_limpio_enriquecido.csv`. La escritura queda registrada en `logs/pipeline.log` junto con las dimensiones iniciales y finales del proceso.

In [6]:
try:
    df.to_csv(OUTPUT_FILE, index=False)
    logger.info("Cantidad final: %s filas x %s columnas", df.shape[0], df.shape[1])
    logger.info("Ruta de guardado: %s", OUTPUT_FILE)
    logger.info("Fin del preprocessing")
except Exception:
    logger.exception("Error durante el guardado del dataset procesado")
    raise

print(f"Dataset procesado guardado en: {OUTPUT_FILE.resolve()}")
print(f"Log actualizado en: {LOG_FILE.resolve()}")

INFO | Cantidad final: 62076 filas x 14 columnas


INFO | Ruta de guardado: C:\Users\Germán\Desktop\TP AVANZADA\tp-final-siniestros-viales\data\processed\siniestros_limpio_enriquecido.csv


INFO | Fin del preprocessing


Dataset procesado guardado en: C:\Users\Germán\Desktop\TP AVANZADA\tp-final-siniestros-viales\data\processed\siniestros_limpio_enriquecido.csv
Log actualizado en: C:\Users\Germán\Desktop\TP AVANZADA\tp-final-siniestros-viales\logs\pipeline.log
